# Step 3: Drivers’ analysis and identification: Criterion 1: Expenditure differentiation assessment (confidence-bound overlap): 

Before running this notebook, please refer to the [Step 3 README](../Step_3/README.md) for instructions on the required input files and folder structure.

This notebook is organized into two main sections:
1. Loading the data files
2. Performing calculations

## Case study

- **Region:** Province of Quebec, Canada
- **Year:** 2021
- **Impact studied:** Climate change short-term
- **Available socioeconomic and demographic (SED) factors:** Income, Household type, Region and Tenure

## Required libraries

In [1]:
import pandas as pd
import itertools

The following parameters define the study year, SED factor, household size of each subfactor and consumption categories. Modify these values for the case study

In [2]:
YEAR  = 2021 
FACTOR = "Income"     
HOUSEHOLDS_PER_FACTOR = ["Q1", "Q2", "Q3", "Q4", "Q5"]
HOUSEHOLD_SIZE = {
            "Q1": 1.22,
            "Q2": 1.67,
            "Q3": 2.04,
            "Q4": 2.93,
            "Q5": 3.51,
        }
CATEGORIES = [
    "Clothing and accessories",
    "Food expenditures",
    "Health and personal care",
    "Household operations, furnishings and, equipment",
    "Miscellaneous expenditures",
    "Recreation",
    "Shelter",
    "Tobacco, alcohol and non-medical cannabis",
    "Transportation"]

## 1. Loading the data files

This file contains the mean expenditure and confidence bounds by SED subfactor and consumption category. It must be consistent with the concordance table used in Step 2.


In [3]:
file_path = f"{YEAR}_{FACTOR}.xlsx"
df = pd.read_excel(file_path)
df

,Segment,Category,Mean_alloc,Lower_alloc,Upper_alloc
0,Q5,Clothing and accessories,3770.000000,3420.000000,4119.000000
1,Q4,Clothing and accessories,2896.000000,2534.000000,3258.000000
2,Q3,Clothing and accessories,1991.000000,1728.000000,2255.000000
3,Q2,Clothing and accessories,1493.000000,1184.000000,1802.000000
4,Q1,Clothing and accessories,763.000000,630.000000,897.000000
5,Q5,Food expenditures,12951.000000,11240.000000,14661.000000
6,Q4,Food expenditures,9491.000000,8281.000000,10700.000000
7,Q3,Food expenditures,6990.000000,6245.000000,7735.000000
8,Q2,Food expenditures,5379.000000,4656.000000,6102.000000
9,Q1,Food expenditures,4653.000000,3831.000000,5475.000000


The per capita expenditure values are calculated by dividing the household expenditure by the average household size of each subfactor.


In [4]:
df["Mean_pp"] = df["Mean_alloc"] / df["Segment"].map(HOUSEHOLD_SIZE)
df["Lower_pp"] = df["Lower_alloc"] / df["Segment"].map(HOUSEHOLD_SIZE)
df["Upper_pp"] = df["Upper_alloc"] / df["Segment"].map(HOUSEHOLD_SIZE)
df

,Segment,Category,Mean_alloc,Lower_alloc,Upper_alloc,Mean_pp,Lower_pp,Upper_pp
0,Q5,Clothing and accessories,3770.000000,3420.000000,4119.000000,1074.074074,974.358974,1173.504274
1,Q4,Clothing and accessories,2896.000000,2534.000000,3258.000000,988.395904,864.846416,1111.945392
2,Q3,Clothing and accessories,1991.000000,1728.000000,2255.000000,975.980392,847.058824,1105.392157
3,Q2,Clothing and accessories,1493.000000,1184.000000,1802.000000,894.011976,708.982036,1079.041916
4,Q1,Clothing and accessories,763.000000,630.000000,897.000000,625.409836,516.393443,735.245902
5,Q5,Food expenditures,12951.000000,11240.000000,14661.000000,3689.743590,3202.279202,4176.923077
6,Q4,Food expenditures,9491.000000,8281.000000,10700.000000,3239.249147,2826.279863,3651.877133
7,Q3,Food expenditures,6990.000000,6245.000000,7735.000000,3426.470588,3061.274510,3791.666667
8,Q2,Food expenditures,5379.000000,4656.000000,6102.000000,3220.958084,2788.023952,3653.892216
9,Q1,Food expenditures,4653.000000,3831.000000,5475.000000,3813.934426,3140.163934,4487.704918


## 2. Performing calculations

Criterion 1 evaluates whether a SED factor produces statistically distinguishable expenditure patterns across its subfactors. For each consumption category, all pairwise combinations of subfactors are compared using their confidence bounds. A factor meets Criterion 1 for a given category if at least one pair of subfactors shows non-overlapping confidence intervals at both the household and per capita levels.


In [5]:
results = []
passing_categories = []

for category in CATEGORIES:
    df_cat = df[df["Category"] == category]
    
    pairs = list(itertools.combinations(HOUSEHOLDS_PER_FACTOR, 2))
    
    pair_rows = []
    sig_alloc_any = False
    sig_pp_any = False
    
    for seg1, seg2 in pairs:
        row1 = df_cat[df_cat["Segment"] == seg1].iloc[0]
        row2 = df_cat[df_cat["Segment"] == seg2].iloc[0]
        
        # Household level
        sig_alloc = (row1["Upper_alloc"] < row2["Lower_alloc"]) or (row2["Upper_alloc"] < row1["Lower_alloc"])
        
        # Per capita level
        sig_pp = (row1["Upper_pp"] < row2["Lower_pp"]) or (row2["Upper_pp"] < row1["Lower_pp"])
        
        if sig_alloc:
            sig_alloc_any = True
        if sig_pp:
            sig_pp_any = True
            
        pair_rows.append({
            "Category": category,
            "Pair": f"{seg1} vs {seg2}",
            "Significant (household)": "Yes" if sig_alloc else "No",
            "Significant (per capita)": "Yes" if sig_pp else "No"
        })
    
    results.extend(pair_rows)
    
    if sig_alloc_any and sig_pp_any:
        passing_categories.append(category)

df_pairs = pd.DataFrame(results)
df_passing = pd.DataFrame({"Category": passing_categories})
                          
df_passing.to_excel(f"{YEAR}_{FACTOR}_Criterion1_summary.xlsx", index=False)
   
df_pairs.to_excel(f"{YEAR}_{FACTOR}_Criterion1_pairs.xlsx", index=False)

print("Categories meeting Criterion 1:")
display(df_passing)
print("\nDetailed pairwise comparison:")
display(df_pairs)

Categories meeting Criterion 1:


,Category
0,Clothing and accessories
1,Miscellaneous expenditures
2,Recreation
3,Shelter



Detailed pairwise comparison:


,Category,Pair,Significant (household),Significant (per capita)
0,Clothing and accessories,Q1 vs Q2,Yes,No
1,Clothing and accessories,Q1 vs Q3,Yes,Yes
2,Clothing and accessories,Q1 vs Q4,Yes,Yes
3,Clothing and accessories,Q1 vs Q5,Yes,Yes
4,Clothing and accessories,Q2 vs Q3,No,No
...,...,...,...,...
85,Transportation,Q2 vs Q4,Yes,No
86,Transportation,Q2 vs Q5,Yes,No
87,Transportation,Q3 vs Q4,Yes,No
88,Transportation,Q3 vs Q5,Yes,No
